<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/bonus/notebooks/8_rnns_and_lstms_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Bonus: Recurrent Neural Networks and LSTMs

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

Before transformers took over in 2018, the standard neural network for text was the **recurrent
neural network** (RNN), and especially its improved version, the **LSTM**. Instead of looking at
a whole text at once, an RNN reads it the way you do: one word at a time, keeping a running
memory of what it has read so far.

RNNs matter for this course for two reasons. They show, very concretely, *why* word order is
information that bag-of-words methods throw away. And their main weakness, forgetting things over
long distances, is exactly the problem that **attention** (Day 3) was invented to solve.

**Contents**

1. Reading with a memory: an RNN by hand
2. When word order matters: a small experiment
3. The memory problem: plain RNNs vs. LSTMs
4. The bill data, as sequences
5. An LSTM classifier in Keras
6. An LSTM classifier in PyTorch
7. Comparing the models
8. From LSTMs to attention and transformers

> A GPU helps but isn't needed: everything here runs on Colab's CPU in a few minutes.

There are 6 exercises along the way, marked ✏️. A separate solutions notebook has worked answers, but try them yourself first.

## 0. Setup

Everything we need comes with Colab except `gensim` (for the GloVe vectors from Day 2), which the
first line installs. As in Bonus 6, we run Keras on the **PyTorch backend**, so both stacks share
one engine. The backend has to be set before Keras is imported.

In [ ]:
!pip install -q gensim

import os
os.environ["KERAS_BACKEND"] = "torch"     # must come before `import keras`

import re
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import keras

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
keras.utils.set_random_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
pd.set_option("display.max_colwidth", 100)
print("Keras", keras.__version__, "on", keras.backend.backend(), "| PyTorch", torch.__version__,
      "| device:", device.upper())

## 1. Reading with a memory: an RNN by hand

An RNN keeps a **hidden state** $h$: a vector that summarizes everything read so far. At every
word it updates that memory, combining the new word with the old memory:

$$h_t = \tanh(W x_t + U h_{t-1} + b)$$

where $x_t$ is the embedding of word $t$, $h_{t-1}$ the memory after the previous word, and $W$,
$U$ and $b$ are weights learned in training. The **same** weights are used at every step, which
is why an RNN can read texts of any length. After the last word, $h$ summarizes the whole text,
and a normal output layer turns it into a prediction.

Here's an RNN with a 4-number memory, reading two sentences made of exactly the same words:

In [ ]:
rng = np.random.default_rng(0)
toy_emb = {w: rng.normal(size=3) for w in ["democrats", "defeated", "republicans"]}  # 3-dim "embeddings"
W = rng.normal(0, 0.5, (4, 3))      # input -> memory
U = rng.normal(0, 0.5, (4, 4))      # old memory -> new memory
b = np.zeros(4)

def read(sentence, verbose=True):
    h = np.zeros(4)                                  # empty memory at the start
    for word in sentence.split():
        h = np.tanh(W @ toy_emb[word] + U @ h + b)   # update the memory with the next word
        if verbose:
            print(f"   after {word!r:<15} h = {h.round(2)}")
    return h

print("Sentence 1:"); h1 = read("democrats defeated republicans")
print("Sentence 2:"); h2 = read("republicans defeated democrats")
print(f"\nSame words, different final memory. Distance between the two: {np.linalg.norm(h1 - h2):.2f}")

A bag of words would represent these two sentences identically. The RNN doesn't, because it
reads in order: the memory after *"democrats defeated"* is different from the memory after
*"republicans defeated"*, and that difference carries through to the end. Whether the
difference is *useful* depends on the weights, and that's what training takes care of.

> **✏️ Exercise 1**
>
> 1. Run `read` on `"defeated democrats republicans"` and compare its final memory with `h1` and
>    `h2`.
> 2. Represent the three sentences as bag-of-words count vectors (you can do it by hand, or with
>    scikit-learn's `CountVectorizer`). How many *different* vectors do you get?

In [ ]:
# Your code here

## 2. When word order matters: a small experiment

Let's build a task that a bag of words *cannot* solve. We generate short sentences about a vote,
such as *"after debate democrats defeated republicans today"*. The label is **who won**: 1 if
the Democrats did, 0 if the Republicans did. Both parties appear in every sentence, so the
words are the same either way; only their **order** gives the answer away.

In [ ]:
import random

FILLER = ["the", "vote", "on", "new", "budget", "bill", "today", "after", "a", "long", "debate",
          "in", "senate", "committee", "late", "tuesday", "night", "final", "round", "state",
          "house", "floor", "session"]
VERBS = ["defeated", "beat", "outvoted"]

def make_vote_data(n, seed, n_before=(0, 3), n_after=(0, 3)):
    # Random filler words, then "<winner> <verb> <loser>", then more filler words
    rnd = random.Random(seed)
    texts, ys = [], []
    for _ in range(n):
        winner, loser = rnd.sample(["democrats", "republicans"], 2)
        words = (rnd.choices(FILLER, k=rnd.randint(*n_before)) + [winner, rnd.choice(VERBS), loser]
                 + rnd.choices(FILLER, k=rnd.randint(*n_after)))
        texts.append(" ".join(words))
        ys.append(int(winner == "democrats"))
    return texts, np.array(ys)

vote_train, vote_y_train = make_vote_data(2000, seed=1)
vote_test, vote_y_test = make_vote_data(500, seed=2)
for t, y in zip(vote_train[:4], vote_y_train[:4]):
    print(f"label {y}:  {t}")

First, the bag-of-words approach from Day 1: word counts and logistic regression.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer()                       # single words only
bow_clf = LogisticRegression(max_iter=1000).fit(bow.fit_transform(vote_train), vote_y_train)
print(f"Bag of words, test accuracy: {accuracy_score(vote_y_test, bow_clf.predict(bow.transform(vote_test))):.3f}")

About 50%: a coin flip. The model sees *democrats*, *republicans* and *defeated* in every sentence
and has no way of telling who did what to whom.

Now a small recurrent network. The class below works for all three kinds of recurrent layers we
discuss (`"rnn"`, `"lstm"` and `"gru"`), so we can reuse it throughout the notebook. Two details
explained in the comments: titles are padded to the same length, so we tell the network each
title's true length (*packing*), so it doesn't read the padding; and we classify from the final
memory.

In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden, n_classes, kind="lstm",
                 bidirectional=False, dropout=0.0, pretrained=None, freeze=False):
        super().__init__()
        if pretrained is None:
            self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        else:
            self.embedding = nn.Embedding.from_pretrained(torch.tensor(pretrained), freeze=freeze, padding_idx=0)
        layer = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[kind]
        self.rnn = layer(emb_dim, hidden, batch_first=True, bidirectional=bidirectional)
        self.kind = kind
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(hidden * (2 if bidirectional else 1), n_classes)

    def forward(self, x):                                   # x: [batch, seq_len] word IDs, 0 = padding
        lengths = (x != 0).sum(dim=1).clamp(min=1).cpu()    # true length of every sequence
        packed = nn.utils.rnn.pack_padded_sequence(         # "packing": skip the padding
            self.embedding(x), lengths, batch_first=True, enforce_sorted=False)
        _, h = self.rnn(packed)                             # h: final memory, [layers*directions, batch, hidden]
        if self.kind == "lstm":
            h = h[0]                                        # an LSTM returns (hidden state, cell state)
        h = torch.cat([h[-2], h[-1]], dim=1) if self.rnn.bidirectional else h[-1]
        return self.out(self.dropout(h))                    # logits

In [ ]:
vote_vocab = {w: i + 1 for i, w in enumerate(sorted(set(FILLER + VERBS + ["democrats", "republicans"])))}

def encode_votes(texts, max_len):
    X = np.zeros((len(texts), max_len), dtype="int64")      # 0 = padding
    for row, t in enumerate(texts):
        ids = [vote_vocab[w] for w in t.split()][:max_len]
        X[row, :len(ids)] = ids
    return X


def fit_vote_model(kind, train_texts, train_y, test_texts, test_y, max_len, epochs=20, seed=SEED):
    # Train a small recurrent classifier on the vote data and return its test accuracy
    torch.manual_seed(seed)
    model = RNNClassifier(len(vote_vocab) + 1, emb_dim=16, hidden=32, n_classes=2, kind=kind)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
    X, y = torch.tensor(encode_votes(train_texts, max_len)), torch.tensor(train_y)
    for epoch in range(epochs):
        order = torch.randperm(len(X))
        for i in range(0, len(X), 32):
            idx = order[i:i + 32]
            loss = nn.functional.cross_entropy(model(X[idx]), y[idx])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(encode_votes(test_texts, max_len))).argmax(dim=1).numpy()
    return accuracy_score(test_y, pred)


acc = fit_vote_model("lstm", vote_train, vote_y_train, vote_test, vote_y_test, max_len=10)
print(f"LSTM, test accuracy: {acc:.3f}")

The recurrent network gets it (close to) perfectly right, from the same words that left the bag
of words guessing. It learned that what comes *before* the verb is the winner.

> **✏️ Exercise 2**
>
> Give the bag-of-words model **word pairs** as well: `CountVectorizer(ngram_range=(1, 2))`. How
> well does it do now, and why? What does this tell you about CNNs (Bonus 7), whose filters are
> learned n-gram detectors?

In [ ]:
# Your code here

## 3. The memory problem: plain RNNs vs. LSTMs

Plain RNNs have a famous weakness. During training, the error signal has to travel backward
through every reading step, and at each step it gets multiplied by the recurrent weights. Over
many steps it tends to shrink toward zero (the **vanishing gradient** problem), so the network
struggles to learn connections between things that are far apart. In practice, a plain RNN mostly
remembers the last few words.

Let's test that. Same task, but now each sentence ends with **8 to 15 filler words** after the
important part, so the network has to carry *who won* through a long stretch of irrelevant words.

In [ ]:
long_train, long_y_train = make_vote_data(2000, seed=1, n_before=(0, 1), n_after=(8, 15))
long_test, long_y_test = make_vote_data(500, seed=2, n_before=(0, 1), n_after=(8, 15))
print("Example:", long_train[0])

for kind in ["rnn", "lstm"]:
    start = time.time()
    acc = fit_vote_model(kind, long_train, long_y_train, long_test, long_y_test, max_len=19)
    print(f"{kind.upper():<5} test accuracy: {acc:.3f}   ({time.time() - start:.0f} s)")

In our runs, the plain RNN usually stays at chance on this version, while the LSTM still solves
it. (Training is random, so you may occasionally see the plain RNN get lucky; Exercise 3 checks
several seeds.)

### How the LSTM remembers

The **LSTM** (*long short-term memory*, Hochreiter & Schmidhuber, 1997) adds a second, protected
memory, the **cell state**, which runs through the whole sequence like a conveyor belt. Three
**gates**, each a small layer with a sigmoid that outputs numbers between 0 (closed) and 1
(open), control it at every word:

| Gate | Decides... |
|---|---|
| **forget gate** | what to erase from the cell state |
| **input gate** | what new information from the current word to write into it |
| **output gate** | what part of the cell state to reveal as the hidden state |

Because information on the conveyor belt is only changed when a gate decides so, the error signal
can travel back through many steps without vanishing. The **GRU** (*gated recurrent unit*, 2014)
is a simpler variant with two gates that often works just as well.

> **✏️ Exercise 3**
>
> Repeat the long-memory experiment for `"rnn"`, `"lstm"` and `"gru"`, each with seeds 1, 2 and
> 3 (`seed=` in `fit_vote_model`), and put the accuracies in a table. Then try a much harder
> version with 30 to 50 filler words (`n_after=(30, 50)`, `max_len=54`). What happens?

In [ ]:
# Your code here

## 4. The bill data, as sequences

Back to a real task.

## The data: US Congressional bills (again)

We use the same data and the **same split** as the Day 3 BERT notebook: bill titles from
`dreamproit/bill_labels_us`, four policy areas (Health, Education, Taxation, Armed Forces and
National Security), 500 bills per area, split 70/15/15 into training, validation and test sets
with the same random seed. That means you can put the scores from this notebook right next to
the TF-IDF baseline and the fine-tuned BERT model from Day 3.

(As on Day 3, we read only two ~200 MB chunks of the dataset's Parquet copy, and only the
columns we need, because the full dataset is a single 5 GB file.)

In [ ]:
from huggingface_hub import hf_hub_download
from sklearn.model_selection import train_test_split

parts = []
for c in [0, 5]:
    path = hf_hub_download(repo_id="dreamproit/bill_labels_us", repo_type="dataset",
                           revision="refs/convert/parquet", filename=f"default/train/{c:04d}.parquet")
    parts.append(pd.read_parquet(path, columns=["title", "policy_area"]))
raw = pd.concat(parts, ignore_index=True)

AREAS = ["Health", "Education", "Taxation", "Armed Forces and National Security"]
clean = (raw.assign(title=raw["title"].fillna("").str.strip())
            .query("title != ''")
            .drop_duplicates(subset="title"))                 # every title once: no leakage
bills = (clean[clean["policy_area"].isin(AREAS)]
         .sample(frac=1, random_state=SEED)
         .groupby("policy_area").head(500)
         .reset_index(drop=True))

labels = sorted(AREAS)
label2id = {name: i for i, name in enumerate(labels)}
bills["label"] = bills["policy_area"].map(label2id)

train_df, temp_df = train_test_split(bills, test_size=0.30, stratify=bills["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)
y_train, y_val, y_test = train_df["label"].values, val_df["label"].values, test_df["label"].values

print(f"train: {len(train_df)}   validation: {len(val_df)}   test: {len(test_df)}")
print("Label IDs:", label2id)

## From text to sequences of numbers

Convolutional and recurrent networks read text **in order**, so we can't use TF-IDF (which
throws the order away). Instead, every title becomes a sequence of word IDs:

1. **Tokenize**: lowercase, split into words.
2. **Build a vocabulary** from the *training* titles only. Words that appear only once are
   mapped to a shared "unknown" token, because the model can't learn anything useful from a
   single example (and so that validation and test words the model never saw are handled the
   same way).
3. **Encode and pad**: replace every word by its ID, and cut or pad every title to the same
   length, so titles can be processed in batches.

This step is the same for both stacks, so we do it once, in plain Python and NumPy.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

PAD, UNK = 0, 1                                   # two reserved IDs
counts = Counter(w for t in train_df["title"] for w in tokenize(t))
vocab = ["<pad>", "<unk>"] + sorted(w for w, c in counts.items() if c >= 2)
word2id = {w: i for i, w in enumerate(vocab)}
print(f"Vocabulary: {len(vocab):,} words (from {len(counts):,} distinct words in the training titles)")

lengths = train_df["title"].apply(lambda t: len(tokenize(t)))
MAX_LEN = int(np.percentile(lengths, 95))         # long enough for 95% of titles
print(f"Title length in words: median {int(lengths.median())}, 95th percentile {MAX_LEN} -> MAX_LEN = {MAX_LEN}")

def encode(texts, max_len=MAX_LEN):
    out = np.full((len(texts), max_len), PAD, dtype="int64")
    for row, text in enumerate(texts):
        ids = [word2id.get(w, UNK) for w in tokenize(text)][:max_len]   # truncate long titles
        out[row, :len(ids)] = ids                                      # pad short ones with 0s
    return out

X_train, X_val, X_test = encode(train_df["title"]), encode(val_df["title"]), encode(test_df["title"])

example = train_df["title"].iloc[0]
print("\nExample title:", example)
print("Encoded:      ", X_train[0])
print("Decoded back: ", " ".join(vocab[i] for i in X_train[0] if i != PAD))

### The baseline to beat

In [ ]:
results = {}

def score(name, y_true, y_pred):
    results[name] = {"accuracy": accuracy_score(y_true, y_pred),
                     "macro-F1": f1_score(y_true, y_pred, average="macro")}
    print(f"{name:<34} validation accuracy {results[name]['accuracy']:.3f} | "
          f"macro-F1 {results[name]['macro-F1']:.3f}")

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
baseline = LogisticRegression(max_iter=1000).fit(tfidf.fit_transform(train_df["title"]), y_train)
score("TF-IDF + logistic regression", y_val, baseline.predict(tfidf.transform(val_df["title"])))

## 5. An LSTM classifier in Keras

Our network:

1. **Embedding** with `mask_zero=True`: Keras's way of saying "ID 0 is padding, ignore it". The
   mask travels along with the data, so the LSTM skips the padded positions (the Keras
   equivalent of packing).
2. **Bidirectional LSTM**: one LSTM reads the title left to right, a second one right to left, and
   their final memories are joined. That way, every part of the title is close to *some*
   reading direction's end, which softens the memory problem from Section 3.
3. **Dropout** and the **output layer**, as always.

In [ ]:
EMB_DIM, HIDDEN, n_classes = 50, 64, len(labels)

keras.utils.set_random_seed(SEED)
keras_lstm = keras.Sequential([
    keras.Input(shape=(MAX_LEN,), dtype="int64"),
    keras.layers.Embedding(len(vocab), EMB_DIM, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(HIDDEN)),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(n_classes, activation="softmax"),
])
keras_lstm.compile(optimizer=keras.optimizers.Adam(1e-3),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
keras_lstm.summary()

An LSTM layer has four sets of weights (three gates plus the candidate memory), which is why it
has roughly four times as many parameters as a plain RNN of the same size.

In [ ]:
start = time.time()
keras_history = keras_lstm.fit(
    X_train, y_train, validation_data=(X_val, y_val), epochs=15, batch_size=32, verbose=0,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)])
print(f"Trained {len(keras_history.history['loss'])} epochs in {time.time() - start:.1f} s")
score("Keras BiLSTM", y_val, keras_lstm.predict(X_val, verbose=0).argmax(axis=1))

In [ ]:
def plot_history(train_loss, val_loss, title):
    plt.figure(figsize=(6.5, 3.2))
    plt.plot(train_loss, label="training loss", color="#1A1A2E")
    plt.plot(val_loss, label="validation loss", color="#34B233")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title(title); plt.legend()
    plt.show()

plot_history(keras_history.history["loss"], keras_history.history["val_loss"], "Keras BiLSTM")

> **✏️ Exercise 4**
>
> Replace the `LSTM` layer with a `GRU` layer (keep it bidirectional). Compare the number of
> parameters, the training time and the validation macro-F1.

In [ ]:
# Your code here

## 6. An LSTM classifier in PyTorch

We reuse the `RNNClassifier` class from Section 2, now bidirectional and with dropout, and the
standard training loop with early stopping.

In [ ]:
def make_loader(X, y, shuffle, batch_size=32):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader = make_loader(X_val, y_val, shuffle=False)
test_loader = make_loader(X_test, y_test, shuffle=False)


def predict_torch(model, loader):
    model.eval()
    preds, total_loss = [], 0.0
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(device))
            total_loss += nn.functional.cross_entropy(logits, yb.to(device)).item() * len(yb)
            preds.append(logits.argmax(dim=1).cpu())
    return torch.cat(preds).numpy(), total_loss / len(loader.dataset)


def train_torch(model, epochs=15, lr=1e-3, patience=3):
    # Standard training loop with early stopping on the validation loss
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    best, best_state, bad_epochs, history = float("inf"), None, 0, {"loss": [], "val_loss": []}
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = loss_fn(model(xb), yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running += loss.item() * len(yb)
        _, val_loss = predict_torch(model, val_loader)
        history["loss"].append(running / len(train_loader.dataset))
        history["val_loss"].append(val_loss)
        if val_loss < best:
            best, best_state, bad_epochs = val_loss, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break
    model.load_state_dict(best_state)
    print(f"Trained {len(history['loss'])} epochs; best validation loss {best:.4f}")
    return history

In [ ]:
torch.manual_seed(SEED)
torch_lstm = RNNClassifier(len(vocab), EMB_DIM, HIDDEN, n_classes, kind="lstm",
                           bidirectional=True, dropout=0.5).to(device)
print(torch_lstm)

start = time.time()
torch_history = train_torch(torch_lstm)
print(f"... in {time.time() - start:.1f} s")
score("PyTorch BiLSTM", y_val, predict_torch(torch_lstm, val_loader)[0])
plot_history(torch_history["loss"], torch_history["val_loss"], "PyTorch BiLSTM")

> **✏️ Exercise 5**
>
> Train a **unidirectional** version (`bidirectional=False`) of the PyTorch LSTM. Is reading in
> both directions worth it for bill titles? Would you expect a bigger or smaller difference for
> long documents?

In [ ]:
# Your code here

### Starting from GloVe

As with the CNN in Bonus 7, we can start the embedding layer from pre-trained GloVe vectors.

In [ ]:
import gensim.downloader as api

glove = api.load("glove-wiki-gigaword-50")      # ~66 MB, the same vectors as on Day 2
EMB_GLOVE = glove.vector_size

# One row per vocabulary word: the GloVe vector if GloVe knows the word, small random numbers if not
rng = np.random.default_rng(SEED)
embedding_matrix = rng.normal(0, 0.1, (len(vocab), EMB_GLOVE)).astype("float32")
embedding_matrix[PAD] = 0.0
found = 0
for word, idx in word2id.items():
    if word in glove.key_to_index:
        embedding_matrix[idx] = glove[word]
        found += 1
print(f"GloVe covers {found:,} of {len(vocab):,} vocabulary words ({found / len(vocab):.0%})")

> **✏️ Exercise 6**
>
> Train the bidirectional PyTorch LSTM starting from `embedding_matrix` (`pretrained=embedding_matrix`,
> with `emb_dim=EMB_GLOVE`), once with `freeze=False` and once with `freeze=True`. How do they
> compare with random embeddings?

In [ ]:
# Your code here

## 7. Comparing the models

In [ ]:
pd.DataFrame(results).T.round(3)

And the test set, once. Same 300 titles as in Day 3 and Bonus 7.

In [ ]:
test_preds = {
    "TF-IDF + logistic regression": baseline.predict(tfidf.transform(test_df["title"])),
    "Keras BiLSTM": keras_lstm.predict(X_test, verbose=0).argmax(axis=1),
    "PyTorch BiLSTM": predict_torch(torch_lstm, test_loader)[0],
}
pd.DataFrame({name: {"test accuracy": accuracy_score(y_test, p),
                     "test macro-F1": f1_score(y_test, p, average="macro")}
              for name, p in test_preds.items()}).T.round(3)

On bill titles, where a few keywords carry most of the signal, LSTMs typically end up close to
the baseline and the CNN, not dramatically above them. Their sensitivity to order and long-range
context pays off more on longer and subtler texts: stance, framing, sarcasm, negation.

## 8. From LSTMs to attention and transformers

The story from here to Day 3 takes three steps.

**1. Translation and the bottleneck (2014).** LSTMs powered the first neural machine
translation systems: one LSTM (the *encoder*) reads the source sentence into its final memory,
and another (the *decoder*) writes the translation from that memory. The weak spot: the whole
sentence must be squeezed into one fixed-size vector, and long sentences don't fit.

**2. Attention (2015).** Bahdanau, Cho and Bengio let the decoder look back at **all** of the
encoder's hidden states and decide, for every word it writes, which source words to focus on. The
memory problem of Section 3 largely disappears, because nothing has to be carried across the
whole sentence anymore.

**3. Attention is all you need (2017).** Vaswani and colleagues dropped the recurrence entirely
and built a model from attention alone: the **transformer**. Without word-by-word reading, all
positions are processed in parallel, which made it possible to train on vastly more text. That's
the architecture behind BERT, which you used on Day 3, and behind today's LLMs.

One more link: **ELMo** (Peters et al., 2018), the first widely used *contextual* embedding
model, was built from bidirectional LSTMs. BERT replaced its LSTMs with transformers a few months
later.

LSTMs haven't vanished. They are still used for time series, speech, small datasets, and devices
with little computing power.

### Further reading

- Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation, 9*(8).
- Cho, K., van Merriënboer, B., Gulcehre, C., Bahdanau, D., Bougares, F., Schwenk, H., & Bengio,
  Y. (2014). Learning phrase representations using RNN encoder-decoder for statistical machine
  translation. *Proceedings of EMNLP 2014*. (Introduces the GRU.)
- Bahdanau, D., Cho, K., & Bengio, Y. (2015). Neural machine translation by jointly learning to
  align and translate. *Proceedings of ICLR 2015*.
- Olah, C. (2015). Understanding LSTM networks. Blog post,
  [colah.github.io/posts/2015-08-Understanding-LSTMs](https://colah.github.io/posts/2015-08-Understanding-LSTMs/).
  The classic illustrated explanation of the gates.
- Peters, M. E., et al. (2018). Deep contextualized word representations. *Proceedings of
  NAACL-HLT 2018*. (ELMo.)